In [5]:
from sortedcontainers import SortedDict
from collections import defaultdict

In [51]:
test = {
    "aaa": 5,
    "bbb": 2,
    "kaa": 3,
}

pretoken_counter = {
    tuple([bytes([b]) for b in key.encode("utf-8")]): value 
    for key, value in test.items()
}

pretoken_counter

{(b'a', b'a', b'a'): 5, (b'b', b'b', b'b'): 2, (b'k', b'a', b'a'): 3}

In [ ]:
def overlap_update(overlap_pair, new_pair, token, pretoken_counter, counter_index, bpe_pairs, pair_to_token):
    """
    A helper function to update the bpe pair count information

    Args:
        overlap_pair: the bytes pair that overlap with the bytes pair to merge
        merge_pair: the bytes pair to merge
        token: the pretoken that contains the overlap_pair and merge_pair
        pretoken_counter: the count of each pertoken
        counter_index: dict mapping count to a list of bpe paris, need to be updated
        bpe_pairs: the count of each bytes pair, need to be updated
    """
    overlap_pair_count = bpe_pairs[overlap_pair]
    counter_index[overlap_pair_count].remove(overlap_pair)
    updated_overlap_pair_count = overlap_pair_count - pretoken_counter[token]
    if updated_overlap_pair_count > 0:
        if updated_overlap_pair_count not in counter_index:
            counter_index[updated_overlap_pair_count] = set()
        counter_index[updated_overlap_pair_count].add(overlap_pair)
        bpe_pairs[overlap_pair] = updated_overlap_pair_count
    
    # update new pair count
    new_pair_count = bpe_pairs[new_pair]
    new_pair_count_updated = new_pair_count + pretoken_counter[token]
    if new_pair_count > 0:
        counter_index[new_pair_count].remove(new_pair)
    if new_pair_count_updated not in counter_index:
        counter_index[new_pair_count_updated] = set()
    counter_index[new_pair_count_updated].add(new_pair)
    bpe_pairs[new_pair] = new_pair_count_updated

    # record the token to this new pair
    pair_to_token[new_pair].add(token)

In [80]:

def update_counter(pair, count, bpe_pairs, counter_index):
    """
    A simple function to update the bpe pair count information
    """
    print(pair, count, bpe_pairs, counter_index)
    current_count = bpe_pairs[pair]
    if current_count > 0:
        counter_index[current_count].remove(pair)
    updated_count = current_count + count
    if updated_count > 0:
        if updated_count not in counter_index:
            counter_index[updated_count] = set()
        counter_index[updated_count].add(pair)
        bpe_pairs[pair] = updated_count
    else:
        del bpe_pairs[pair]

def merge_token(token, bpe_pair, indices):
    merged_token = []
    merged_pair = (bpe_pair[0] + bpe_pair[1], )
    merged_token = tuple()
    for idx, i in enumerate(indices):
        if idx == 0:
            merged_token += token[:i]
            merged_token += merged_pair
        elif idx > 0 and (indices[idx] - indices[idx-1]) == 1:
            merged_token += (token[i], )  # no merge
        else:
            merged_token += token[indices[idx-1]+2:i]
            merged_token += merged_pair
    # merge the remaining part
    merged_token += token[indices[-1]+2:]
    return merged_token

merge_token(tuple(bytes([b]) for b in "lower".encode("utf-8")), (b'o', b'w'), [1])
print(merge_token(tuple(bytes([b]) for b in "aaaa".encode("utf-8")), (b'a', b'a'), [0, 2]))

def update_bpe_pairs(token, merged_token, token_count, bpe_pairs, counter_index, pair_to_token):
    """
    A naive implementation to compute the new token after merge and update the bpe pairs
    """
    orig_count = defaultdict(int)
    for l, r in zip(token[:-1], token[1:]):
        orig_count[(l, r)] += token_count

    new_count = defaultdict(int)
    if len(merged_token) > 1:  # still pairs available
        for l, r in zip(merged_token[0:], merged_token[1:]):
            new_count[(l, r)] += token_count
    
    # start to compute update
    for k, v in orig_count.items():
        update_counter(k, -v, bpe_pairs, counter_index)
    for k, v in new_count.items():
        update_counter(k, v, bpe_pairs, counter_index)
        if k not in pair_to_token:
            pair_to_token[k].add(token)


(b'aa', b'aa')


In [82]:
vocab = {i: bytes([i]) for i in range(256)}  # the initial vocab
# for special_token in special_tokens:  # assign id to special tokens
#     vocab[special_token.encode("utf-8")] = len(vocab)

bpe_pairs_merged = []  # the bpe pairs that get merged
bpe_pairs = defaultdict(int)  # the counter of bpe pairs
counter_index = SortedDict()
pair_to_token = defaultdict(set)  # from token to bpe back tracking
# construct the initial bpe pair counts
for key, value in pretoken_counter.items():
    for l, r in zip(key[:-1], key[1:]):
        bpe_pairs[(l, r)] += value
        pair_to_token[(l, r)].add(key)
# keep a record of the original pretoken and the merged pretoken
original_to_merged = {
    key: key for key in pretoken_counter
}

# index the counts
for pair, count in bpe_pairs.items():
    if count not in counter_index:
        counter_index[count] = set()
    counter_index[count].add(pair)

for i in range(5):  # loop until we merge sufficient number of tokens
    # retrieval the current largest count and the pairs
    while True:
        top_count, top_bpe_pairs = counter_index.peekitem()
        if len(top_bpe_pairs) > 0:
            break
        del counter_index[top_count]
    print(f"top count: {top_count}, top_bpe_paris: {top_bpe_pairs}")
    top_bpe_pairs = sorted(list(top_bpe_pairs), key=lambda x: x[0]+x[1])
    bpe_pair = top_bpe_pairs[-1]  # select the largest one to merge
    bpe_pairs_merged.append(bpe_pair)
    vocab[bpe_pair[0] + bpe_pair[1]] = len(vocab)
    # incrementally update the pairs that overlapped
    # I haven't come up with a very good solution here yet, but a relative brute force
    # solution is to:
    #   1. first find the original pretoken that is affected
    #   2. use the original pretoken to find the merged pretoken
    #   3. loop through the pretoken and identify the bpe pairs that overlap
    #   4. update the counter
    tokens = pair_to_token[bpe_pair]  # tokens need to be merged
    for token in tokens:  # each token is a tuple of bytes
        current_token = original_to_merged[token]
        print(current_token)
        # need to find all starting index of the merged pair
        indices = []
        for i in range(len(current_token) - 1):
            if current_token[i] == bpe_pair[0] and current_token[i+1] == bpe_pair[1]:
                indices.append(i)

        print(indices)
        filtered_indices = []
        for i in indices:
            if not filtered_indices or i - 1 > filtered_indices[-1]:
                filtered_indices.append(i)
        print(f"after filter: {filtered_indices}")

        merged_token = merge_token(current_token, bpe_pair, indices)
        print(f"token before {current_token}, after merged {merged_token}")
        original_to_merged[token] = merged_token

        update_bpe_pairs(
            current_token, 
            merged_token, 
            pretoken_counter[token], 
            bpe_pairs, 
            counter_index, 
            pair_to_token,
        )
    
    counter_index[top_count] = set(top_bpe_pairs[:-1])
    # clean up the merged bpe pair
    del pair_to_token[bpe_pair]

    print("======== After update ========")
    print(bpe_pairs)

bpe_pairs_merged

top count: 13, top_bpe_paris: {(b'a', b'a')}
(b'a', b'a', b'a')
[0, 1]
after filter: [0]
token before (b'a', b'a', b'a'), after merged (b'aa', b'a')
(b'a', b'a') -10 defaultdict(<class 'int'>, {(b'a', b'a'): 13, (b'b', b'b'): 4, (b'k', b'a'): 3}) SortedDict({3: {(b'k', b'a')}, 4: {(b'b', b'b')}, 13: {(b'a', b'a')}})
(b'aa', b'a') 5 defaultdict(<class 'int'>, {(b'a', b'a'): 3, (b'b', b'b'): 4, (b'k', b'a'): 3}) SortedDict({3: {(b'k', b'a'), (b'a', b'a')}, 4: {(b'b', b'b')}, 13: set()})
(b'k', b'a', b'a')
[1]
after filter: [1]
token before (b'k', b'a', b'a'), after merged (b'k', b'aa')
(b'k', b'a') -3 defaultdict(<class 'int'>, {(b'a', b'a'): 3, (b'b', b'b'): 4, (b'k', b'a'): 3, (b'aa', b'a'): 5}) SortedDict({3: {(b'k', b'a'), (b'a', b'a')}, 4: {(b'b', b'b')}, 5: {(b'aa', b'a')}, 13: set()})
(b'a', b'a') -3 defaultdict(<class 'int'>, {(b'a', b'a'): 3, (b'b', b'b'): 4, (b'aa', b'a'): 5}) SortedDict({3: {(b'a', b'a')}, 4: {(b'b', b'b')}, 5: {(b'aa', b'a')}, 13: set()})
(b'k', b'aa') 3 defa

[(b'a', b'a'), (b'aa', b'a'), (b'b', b'b'), (b'k', b'aa'), (b'bb', b'b')]

In [56]:
def merge_token(token, bpe_pair, indices):
    merged_token = []
    merged_pair = (bpe_pair[0] + bpe_pair[1], )
    merged_token = tuple()
    for idx, i in enumerate(indices):
        if idx == 0:
            merged_token += token[:i]
            merged_token += merged_pair
        elif idx > 0 and (indices[idx] - indices[idx-1]) == 1:
            merged_token += token[i]  # no merge
        else:
            merged_token += token[indices[idx-1]+2:i]
            merged_token += merged_pair
    # merge the remaining part
    merged_token += token[indices[-1]+2:]
    return merged_token

merge_token(tuple(bytes([b]) for b in "lower".encode("utf-8")), (b'o', b'w'), [1])

(b'l', b'ow', b'e', b'r')